# Atividade 1 – Protótipo de reconhecimento facial para os centros de retirada
**Melhores Compras – Fase 6 | Grupo 86**

**Objetivo:** provar que o box de retirada pode liberar o acesso **sem funcionário**, reconhecendo o rosto do cliente.
Usamos 5 pessoas com 5 fotos cada: **4 fotos de treino** (o "cadastro" do cliente) e **1 foto de teste**
(o cliente chegando ao box). O modelo precisa acertar quem é a pessoa da 5ª foto.

**Abordagem escolhida (a "mais simples" sugerida no enunciado):** comparar a foto de teste com as fotos de treino
por **similaridade de cosseno**, usando redes pré-treinadas:
1. **Detecção do rosto** – YuNet (CNN leve do OpenCV Model Zoo). Mostramos também o Haar Cascade da Figura 1.
2. **Extração de características** – SFace, que transforma cada rosto num vetor (embedding) de 128 números.
3. **Comparação** – similaridade de cosseno entre o vetor do teste e o de cada pessoa cadastrada.

### Estrutura de pastas esperada
```
imagem/
├── ruan/
│   ├── treino/   (4 fotos)
│   └── teste/    (1 foto)
├── celebridade_1/
│   ├── treino/
│   └── teste/
...
```


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Bibliotecas
import os, glob, zipfile, urllib.request
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd
!pip install -q opencv-python-headless==4.10.0.84

print('OpenCV:', cv2.__version__)   # YuNet e SFace exigem OpenCV >= 4.5.4
assert hasattr(cv2, 'FaceDetectorYN') and hasattr(cv2, 'FaceRecognizerSF'), 'Atualize o opencv-python'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.0 MB/s eta 0:00:00
OpenCV: 4.14.0


In [3]:
# imagem
import os, zipfile

PASTA = 'imagem'

# Caminhos possíveis para o arquivo zip
caminhos_zip = [
    'imagem.zip',
    '/content/drive/MyDrive/imagem.zip'  # Tenta buscar também no Google Drive montado
]

if not os.path.isdir(PASTA):
    zip_encontrado = False
    for caminho in caminhos_zip:
        if os.path.exists(caminho):
            print(f'Descompactando {caminho}...')
            zipfile.ZipFile(caminho).extractall('.')
            zip_encontrado = True
            break
    if not zip_encontrado:
        raise FileNotFoundError("O diretório 'imagem' não existe e o arquivo 'imagem.zip' não foi encontrado localmente ou no seu Google Drive. Por favor, faça o upload do arquivo 'imagem.zip'.")

EXT = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
def lista(pasta):
    return sorted(f for f in glob.glob(os.path.join(pasta, '*')) if f.lower().endswith(EXT))

pessoas = sorted(p for p in os.listdir(PASTA) if os.path.isdir(os.path.join(PASTA, p)))
base = {p: {'treino': lista(os.path.join(PASTA, p, 'treino')),
            'teste':  lista(os.path.join(PASTA, p, 'teste'))} for p in pessoas}

resumo = pd.DataFrame({p: {k: len(v) for k, v in d.items()} for p, d in base.items()}).T
print(resumo)
assert len(pessoas) == 5, 'O enunciado pede 5 pessoas'
assert (resumo['treino'] == 4).all() and (resumo['teste'] == 1).all(), 'Cada pessoa: 4 fotos de treino e 1 de teste'

Descompactando /content/drive/MyDrive/imagem.zip...
               treino  teste
BradPitt            4      1
Lula                4      1
MarieleFranco       4      1
Rozana              4      1
Ruan                4      1


In [4]:
# Visualizar todas as fotos (linha = pessoa; a última coluna é a de teste )
def ler_rgb(caminho):
    img = cv2.imread(caminho)                   # o OpenCV lê em BGR
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(len(pessoas), 5, figsize=(13, 2.8 * len(pessoas)))
for i, p in enumerate(pessoas):
    for j, f in enumerate(base[p]['treino'] + base[p]['teste']):
        axes[i, j].imshow(ler_rgb(f)); axes[i, j].axis('off')
        axes[i, j].set_title(f"{p} – {'TESTE' if j == 4 else f'treino {j+1}'}", fontsize=9)
plt.tight_layout(); plt.show()

## 2. Detecção do rosto
Todo reconhecimento facial começa pela **detecção**: achar onde está o rosto e recortá-lo.

**2.1 Haar Cascade (Viola-Jones, 2001)**


In [5]:
# Haar Cascade nas fotos de teste
if hasattr(cv2, 'CascadeClassifier'):
    haar = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    fig, axes = plt.subplots(1, len(pessoas), figsize=(16, 4))
    for ax, p in zip(axes, pessoas):
        img = cv2.imread(base[p]['teste'][0])
        cinza = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = haar.detectMultiScale(cinza, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
        for (x, y, w, h) in faces:
            cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), max(2, img.shape[1] // 200))
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.axis('off')
        ax.set_title(f'{p}: {len(faces)} rosto(s)')
    plt.suptitle('Haar Cascade'); plt.tight_layout(); plt.show()
else:
    print('Haar Cascade indisponível nesta versão do OpenCV; seguimos direto com o YuNet.')

**2.2 YuNet** – uma CNN baseada na MobileNetV2, com só ~76 mil parâmetros, que roda em ~1,6 ms por imagem em CPU
e é bem mais precisa que o Haar. Além da caixa do rosto, devolve **5 pontos faciais** (olhos, nariz e cantos da boca),
que o SFace usa para **alinhar** o rosto antes de compará-lo. É a que usamos no protótipo.

In [6]:
# Baixar os modelos pré-treinados (OpenCV Model Zoo, formato ONNX)
MODELOS = {
    'face_detection_yunet_2023mar.onnx':
        'https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx',
    'face_recognition_sface_2021dec.onnx':
        'https://github.com/opencv/opencv_zoo/raw/main/models/face_recognition_sface/face_recognition_sface_2021dec.onnx',
}
for nome, url in MODELOS.items():
    if not os.path.exists(nome):
        urllib.request.urlretrieve(url, nome)
    print(f'{nome}: {os.path.getsize(nome) / 1e6:.1f} MB')

detector = cv2.FaceDetectorYN.create('face_detection_yunet_2023mar.onnx', '', (320, 320),
                                     score_threshold=0.8, nms_threshold=0.3, top_k=5000)
reconhecedor = cv2.FaceRecognizerSF.create('face_recognition_sface_2021dec.onnx', '')

face_detection_yunet_2023mar.onnx: 0.2 MB
face_recognition_sface_2021dec.onnx: 38.7 MB


In [7]:
#Funções: detectar o rosto principal e gerar o embedding
def carregar(caminho, lado_max=1280):
    # Fotos de celular são enormes; reduzir acelera a detecção sem perder qualidade relevante.
    img = cv2.imread(caminho)
    if img is None:
        raise ValueError(f'Não consegui abrir {caminho} (arquivo corrompido ou formato não suportado).')
    escala = lado_max / max(img.shape[:2])
    if escala < 1:
        img = cv2.resize(img, None, fx=escala, fy=escala, interpolation=cv2.INTER_AREA)
    return img

def _detectar(img, limiar):
    h, w = img.shape[:2]
    detector.setInputSize((w, h))
    detector.setScoreThreshold(limiar)
    _, faces = detector.detect(img)      # cada linha: x, y, w, h, 10 coordenadas de landmarks, confiança
    if faces is None or len(faces) == 0:
        return None
    return faces[np.argmax(faces[:, 2] * faces[:, 3])]   # se houver mais de um rosto, fica o MAIOR

def detectar(img):
    # Tenta várias versões da imagem até achar o rosto. Devolve (imagem usada, rosto).
    h, w = img.shape[:2]
    b = int(0.25 * max(h, w))
    versoes = [
        img,                                                                   # original
        cv2.copyMakeBorder(img, b, b, b, b, cv2.BORDER_CONSTANT, value=(0, 0, 0)),  # close: rosto colado na borda
        cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC) if max(h, w) < 800 else None,  # foto pequena
        cv2.resize(img, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA),   # rosto muito grande
    ]
    for limiar in (0.8, 0.6, 0.4):        # começa exigente e afrouxa só se precisar
        for v in versoes:
            if v is None:
                continue
            face = _detectar(v, limiar)
            if face is not None:
                return v, face
    return None, None

def embedding(caminho):
    img, face = detectar(carregar(caminho))
    if face is None:
        raise ValueError(f'Nenhum rosto detectado em {caminho}. Troque essa foto por outra com o rosto mais visível.')
    alinhado = reconhecedor.alignCrop(img, face)       # alinha pelos olhos e recorta 112x112
    vetor = reconhecedor.feature(alinhado).flatten()   # vetor de 128 características
    return vetor / np.linalg.norm(vetor), img, face, alinhado

# Visualizar a detecção do YuNet nas fotos de teste e o rosto alinhado que vai para o SFace
fig, axes = plt.subplots(2, len(pessoas), figsize=(16, 7))
for j, p in enumerate(pessoas):
    _, img, face, alinhado = embedding(base[p]['teste'][0])
    x, y, w, h = face[:4].astype(int)
    vis = img.copy()
    cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), max(2, vis.shape[1] // 200))
    for k in range(5):                                  # 5 landmarks
        cv2.circle(vis, (int(face[4 + 2*k]), int(face[5 + 2*k])), max(3, vis.shape[1] // 250), (0, 0, 255), -1)
    axes[0, j].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); axes[0, j].axis('off')
    axes[0, j].set_title(f'{p} (conf. {face[-1]:.2f})')
    axes[1, j].imshow(cv2.cvtColor(alinhado, cv2.COLOR_BGR2RGB)); axes[1, j].axis('off')
    axes[1, j].set_title('rosto alinhado')
plt.suptitle('YuNet: detecção + landmarks'); plt.tight_layout(); plt.show()

## 3. "Treino": cadastro com as 4 fotos
Na abordagem por similaridade não ajustamos pesos: as redes já vêm treinadas em milhões de rostos.
"Treinar" é **cadastrar** cada pessoa, guardando os vetores das 4 fotos de treino. Também criamos um
**vetor-protótipo** por pessoa (média dos 4 vetores), que resume o rosto dela e reduz o efeito de uma foto ruim.

Isso é **one-shot / few-shot learning**: poucas fotos por pessoa bastam, o que é essencial no mundo real
(um cliente não vai mandar 500 selfies). Treinar uma rede do zero com 4 fotos por pessoa causaria overfitting.

In [8]:
# Cadastro (embeddings de treino)
cadastro, prototipos = {}, {}
for p in pessoas:
    vetores = np.array([embedding(f)[0] for f in base[p]['treino']])
    cadastro[p] = vetores
    media = vetores.mean(axis=0)
    prototipos[p] = media / np.linalg.norm(media)

# Consistência interna: o quanto as 4 fotos da mesma pessoa se parecem entre si
for p in pessoas:
    s = cadastro[p] @ cadastro[p].T
    print(f'{p:>20}: similaridade média entre as fotos de treino = {s[np.triu_indices(4, 1)].mean():.3f}')

            BradPitt: similaridade média entre as fotos de treino = 0.812
                Lula: similaridade média entre as fotos de treino = 0.529
       MarieleFranco: similaridade média entre as fotos de treino = 0.708
              Rozana: similaridade média entre as fotos de treino = 0.753
                Ruan: similaridade média entre as fotos de treino = 0.471


## 4. Reconhecimento da 5ª foto
- **Similaridade de cosseno** (ângulo entre os vetores): quanto mais perto de 1, mais parecidos.
  O limiar oficial do SFace para "mesma pessoa" é **0,363**.
- **Distância euclidiana (L2)** entre vetores normalizados: quanto mais perto de 0, mais parecidos.
  Limiar oficial: **1,128**.

A pessoa prevista é a de **maior similaridade com o protótipo**. Se nem a melhor passar do limiar,
a resposta é **"Desconhecido"** e o box não abre (essencial para a segurança).

In [9]:
# Comparar cada foto de teste com cada pessoa cadastrada
LIMIAR_COS, LIMIAR_L2 = 0.363, 1.128

nomes_proto = list(prototipos)
P = np.array([prototipos[p] for p in nomes_proto])
linhas, matriz = [], []
for real in pessoas:
    v = embedding(base[real]['teste'][0])[0]
    cos = P @ v                                    # cosseno com o protótipo de cada pessoa
    matriz.append(cos)
    melhor = int(np.argmax(cos))
    l2 = np.linalg.norm(P[melhor] - v)
    previsto = nomes_proto[melhor] if cos[melhor] >= LIMIAR_COS else 'Desconhecido'
    # 2ª opinião: a melhor similaridade com as FOTOS individuais (não com a média)
    melhor_foto = max(nomes_proto, key=lambda p: (cadastro[p] @ v).max())
    linhas.append({'pessoa real': real, 'previsto': previsto, 'cosseno': cos[melhor],
                   'distância L2': l2, '2º mais parecido': nomes_proto[np.argsort(cos)[-2]],
                   'cosseno do 2º': np.sort(cos)[-2], 'previsto por foto': melhor_foto,
                   'acertou': previsto == real})

resultado = pd.DataFrame(linhas)
print(resultado.round(3).to_string(index=False))
print(f"\nAcertos: {resultado['acertou'].sum()} de {len(resultado)}")

  pessoa real      previsto  cosseno  distância L2 2º mais parecido  cosseno do 2º previsto por foto  acertou
     BradPitt      BradPitt    0.817         0.605           Rozana          0.210          BradPitt     True
         Lula          Lula    0.682         0.798         BradPitt          0.093              Lula     True
MarieleFranco MarieleFranco    0.793         0.644         BradPitt          0.209     MarieleFranco     True
       Rozana        Rozana    0.803         0.628         BradPitt          0.261            Rozana     True
         Ruan          Ruan    0.766         0.684    MarieleFranco          0.136              Ruan     True

Acertos: 5 de 5


In [10]:
# Matriz de similaridade (teste x cadastro)
M = np.array(matriz)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(M, cmap='RdYlGn', vmin=-0.2, vmax=1)
ax.set_xticks(range(len(nomes_proto)), nomes_proto, rotation=30, ha='right')
ax.set_yticks(range(len(pessoas)), [f'teste: {p}' for p in pessoas])
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center',
                weight='bold' if M[i, j] >= LIMIAR_COS else 'normal')
plt.colorbar(im, label='similaridade de cosseno')
ax.set_title(f'Diagonal verde = acerto | limiar = {LIMIAR_COS}')
plt.tight_layout(); plt.show()

In [11]:
# O que o box "veria"
fig, axes = plt.subplots(1, len(pessoas), figsize=(16, 4.5))
for ax, (_, r) in zip(axes, resultado.iterrows()):
    _, img, face, _ = embedding(base[r['pessoa real']]['teste'][0])
    x, y, w, h = face[:4].astype(int)
    cor = (0, 200, 0) if r['acertou'] else (0, 0, 255)
    cv2.rectangle(img, (x, y), (x + w, y + h), cor, max(2, img.shape[1] // 150))
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.axis('off')
    status = 'ACESSO LIBERADO' if r['previsto'] != 'Desconhecido' else 'ACESSO NEGADO'
    ax.set_title(f"{r['previsto']} ({r['cosseno']:.2f})\n{status}",
                 color='green' if r['acertou'] else 'red', fontsize=10)
plt.tight_layout(); plt.show()

## 5. Teste de robustez (validação cruzada)
Um acerto em 5 fotos pode ser sorte. Para avaliar melhor, repetimos o experimento **5 vezes**: em cada rodada,
uma foto diferente de cada pessoa vira a de teste e as outras 4 viram o cadastro. São 25 reconhecimentos no total.

In [12]:
# Leave-one-out: cada uma das 5 fotos é teste uma vez
todos = {p: np.array([embedding(f)[0] for f in base[p]['treino'] + base[p]['teste']]) for p in pessoas}

registros = []
for rodada in range(5):
    protos = {}
    for p in pessoas:
        treino = np.delete(todos[p], rodada, axis=0)
        m = treino.mean(axis=0); protos[p] = m / np.linalg.norm(m)
    Pm = np.array([protos[p] for p in pessoas])
    for real in pessoas:
        cos = Pm @ todos[real][rodada]
        k = int(np.argmax(cos))
        prev = pessoas[k] if cos[k] >= LIMIAR_COS else 'Desconhecido'
        registros.append({'rodada': rodada + 1, 'real': real, 'previsto': prev,
                          'cosseno': cos[k], 'acertou': prev == real})

cv = pd.DataFrame(registros)
print(f"Acurácia na validação cruzada: {cv['acertou'].mean():.0%} ({cv['acertou'].sum()} de {len(cv)})")
print(pd.crosstab(cv['real'], cv['previsto']))
erros = cv[~cv['acertou']]
if len(erros):
    print('\nErros (vale olhar essas fotos):'); print(erros.round(3).to_string(index=False))

Acurácia na validação cruzada: 100% (25 de 25)
previsto       BradPitt  Lula  MarieleFranco  Rozana  Ruan
real                                                      
BradPitt              5     0              0       0     0
Lula                  0     5              0       0     0
MarieleFranco         0     0              5       0     0
Rozana                0     0              0       5     0
Ruan                  0     0              0       0     5


## 6. Conclusão
**Algoritmo escolhido:** pipeline com dois modelos de deep learning pré-treinados (CNNs do OpenCV Model Zoo). O **YuNet** detecta o rosto e 5 pontos faciais; o **SFace** alinha o rosto e o transforma num vetor de 128 características. Cada pessoa é cadastrada com o vetor médio das 4 fotos de treino, e a foto de teste é atribuída à pessoa com maior **similaridade de cosseno**, desde que acima do limiar de 0,363; abaixo disso o acesso é negado. Não treinamos uma rede própria porque 4 fotos por pessoa são poucas para treinar do zero sem overfitting, enquanto as redes pré-treinadas já aprenderam a diferenciar rostos a partir de milhões de imagens.

**Resultado:** **5 de 5** fotos de teste reconhecidas corretamente; na validação cruzada, **100% (25 de 25)**. A pessoa correta teve cosseno entre **0,682 e 0,818**, enquanto a segunda mais parecida ficou entre **0,092 e 0,261**, sempre abaixo do limiar, o que mostra uma margem de segurança ampla.

**Foi bem-sucedido?** Sim, o protótipo mostra que a liberação automática é viável. Para produção, recomendamos:
- **Prova de vida (anti-spoofing)**, para impedir o uso de foto ou vídeo de outra pessoa.
- **Calibrar o limiar** com mais pessoas, equilibrando falso aceite e falsa rejeição.
- **Plano B** sem funcionário: código QR ou PIN quando o rosto não for reconhecido.
- **LGPD:** biometria é dado pessoal sensível; exige consentimento, armazenamento só dos vetores com criptografia e opção de exclusão.
- Testar o desempenho em diferentes tons de pele, idades e condições de luz, para evitar viés.